# 🪷 AI Monk — SadTalker + LivePortrait Pipeline

Full pipeline: Monk photo → SadTalker (audio-driven lip-sync) → LivePortrait (motion refinement) → final talking video

**Workflow:** Photo + Audio → SadTalker → LivePortrait → talking_monk.mp4

**Setup time:** ~8 min (first run, model downloads). 
**Generation time:** ~3-5 min per clip on T4.

---

## Steps

1. **Runtime → Change runtime type → T4 GPU** (free, 15GB VRAM)
2. Run **Cell 1** — Download monk photo (auto-downloaded, no upload needed)
3. Run **Cell 2** — Install SadTalker (~3 min)
4. Run **Cell 3** — Install LivePortrait (~30 sec)
5. Run **Cell 4** — Download pretrained weights (~3-4 min, one-time only)
6. Run **Cell 5** — Generate SadTalker video (photo + audio → talking head)
7. Run **Cell 6** — Run LivePortrait (refine SadTalker output)
8. Run **Cell 7** — Preview + download final video

Once downloaded to your Mac → use it as the asset for AI Monk reels.

## Cell 1 — Download Monk Photo

Downloads the AI Monk photo (generated via Gemini). No upload needed — it's fetched from GitHub.
If the URL fails, you can upload manually when prompted.

In [ ]:
import os, subprocess

os.makedirs('/content/assets', exist_ok=True)

# Download monk photo from GitHub raw URL
MONK_URL = 'https://raw.githubusercontent.com/charliechan/openavatar_studio/main/monk-1.jpg'
MONK_PATH = '/content/assets/monk.jpg'

print('⏬ Downloading monk photo...')
result = subprocess.run(['curl', '-sL', '-o', MONK_PATH, MONK_URL], capture_output=True, text=True)
if os.path.exists(MONK_PATH) and os.path.getsize(MONK_PATH) > 1000:
    size = os.path.getsize(MONK_PATH) // 1024
    print(f'✅ Monk photo downloaded: {MONK_PATH} ({size}KB)')
else:
    print('⚠️  Auto-download failed. Please upload monk.jpg manually:')
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(f'/content/assets/{name}', 'wb') as f:
            f.write(data)
        MONK_PATH = f'/content/assets/{name}'
        print(f'  ✅ Saved {MONK_PATH}')

PHOTO = MONK_PATH
print(f'\n📷 Source photo: {PHOTO}')

## Cell 2 — Install SadTalker (~3 min)

SadTalker generates a talking-head video from a photo + audio.
Output becomes the driving video for LivePortrait.

In [ ]:
import os, sys, subprocess

SAD_DIR = '/content/SadTalker'
if not os.path.exists(SAD_DIR):
    print('⏬ Cloning SadTalker...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/OpenTalker/SadTalker.git', SAD_DIR], check=True)
else:
    print('✅ SadTalker already cloned')

os.chdir(SAD_DIR)
print('📦 Installing SadTalker requirements...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.0.1', 'torchvision==0.15.2', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('✅ SadTalker install complete')

## Cell 3 — Install LivePortrait

In [ ]:
import os, sys, subprocess

LP_DIR = '/content/LivePortrait'
if not os.path.exists(LP_DIR):
    print('⏬ Cloning LivePortrait…')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/KwaiVGI/LivePortrait.git', LP_DIR], check=True)
else:
    print('✅ LivePortrait already cloned')

os.chdir(LP_DIR)
print('📦 Installing LivePortrait requirements...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'av'], check=True)
print('✅ LivePortrait install complete')

## Cell 4 — Download Pretrained Weights (~4 min, one-time)

Downloads weights for both SadTalker (~2GB) and LivePortrait (~1GB).

In [ ]:
import os, subprocess, sys

# --- SadTalker weights ---
print('⏬ Downloading SadTalker weights...')
os.chdir('/content/SadTalker')
subprocess.run(['bash', 'scripts/download_models.sh'], capture_output=True, text=True)
print('✅ SadTalker weights done')

# --- LivePortrait weights ---
print('⏬ Downloading LivePortrait weights (~1GB)...')
os.chdir('/content/LivePortrait')
if not os.path.exists('/content/LivePortrait/pretrained_weights/liveportrait'):
    subprocess.run(['python3', 'scripts/download_models.py'], check=True)
    print('✅ LivePortrait weights done')
else:
    print('✅ LivePortrait weights already present')

print('\n✅ All weights downloaded!')

## Cell 5 — Generate SadTalker Video (Photo + Audio → Talking Head)

Upload an audio file (.wav or .mp3) with the monk's speech. If no audio is uploaded, a placeholder TTS audio will be generated.

In [ ]:
import os, subprocess, sys, glob

os.makedirs('/content/sadtalker_output', exist_ok=True)

# --- Get audio: upload or generate placeholder ---
AUDIO_PATH = None
existing_audio = glob.glob('/content/assets/*.wav') + glob.glob('/content/assets/*.mp3')
if existing_audio:
    AUDIO_PATH = existing_audio[0]
    print(f'📎 Using uploaded audio: {AUDIO_PATH}')
else:
    print('⬆️  Upload an audio file (.wav/.mp3) with the monk speech:')
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        path = f'/content/assets/{name}'
        with open(path, 'wb') as f:
            f.write(data)
        AUDIO_PATH = path
        print(f'  ✅ Saved {path}')

if not AUDIO_PATH:
    # Generate placeholder TTS audio using gTTS
    print('🔊 No audio uploaded. Generating placeholder TTS...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gTTS'], check=True)
    from gtts import gTTS
    tts = gTTS('Namo Amitabha. May all beings be happy and free from suffering.', lang='en')
    AUDIO_PATH = '/content/assets/monk_speech.wav'
    tts.save(AUDIO_PATH)
    print(f'  ✅ Placeholder audio: {AUDIO_PATH}')

print(f'\n🎙️  Audio: {AUDIO_PATH}')
print(f'📷 Photo: {PHOTO}')

# --- Run SadTalker ---
print('\n🎬 Running SadTalker (photo + audio → talking head)...')
os.chdir('/content/SadTalker')
cmd = [
    sys.executable, 'inference.py',
    '--driven_audio', AUDIO_PATH,
    '--source_image', PHOTO,
    '--result_dir', '/content/sadtalker_output',
    '--still',
    '--preprocess', 'crop',
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

# Find SadTalker output video
sad_videos = glob.glob('/content/sadtalker_output/**/*.mp4', recursive=True)
if sad_videos:
    SAD_VIDEO = sad_videos[0]
    size = os.path.getsize(SAD_VIDEO) // 1024
    print(f'\n✅ SadTalker output: {SAD_VIDEO} ({size}KB)')
else:
    print('❌ SadTalker did not produce a video. Check errors above.')
    SAD_VIDEO = None

## Cell 6 — Run LivePortrait (Refine SadTalker Output)

Uses the SadTalker video as the driving video for LivePortrait. The monk photo gets the SadTalker head motion transferred onto it.

In [ ]:
import subprocess, os, glob

try:
    SAD_VIDEO
except NameError:
    SAD_VIDEO = None

if not SAD_VIDEO or not os.path.exists(SAD_VIDEO):
    vids = glob.glob('/content/sadtalker_output/**/*.mp4', recursive=True)
    if vids:
        SAD_VIDEO = vids[0]
    else:
        raise SystemExit('❌ No SadTalker video found. Run Cell 5 first.')

OUT_DIR = '/content/output'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'\n🎬 Running LivePortrait')
print(f'  source photo : {PHOTO}')
print(f'  driving video: {SAD_VIDEO}')
print(f'  output dir  : {OUT_DIR}\n')

os.chdir('/content/LivePortrait')
cmd = [
    'python3', 'inference.py',
    '-s', PHOTO,
    '-d', SAD_VIDEO,
    '-o', OUT_DIR,
]

result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

print('\n📁 Output files:')
for root, _, files_ in os.walk(OUT_DIR):
    for f in files_:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f'  {size:.2f}MB  {path}')

## Cell 7 — Preview Final Video + Download

In [ ]:
import os, glob
from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

mp4s = glob.glob('/content/output/**/*.mp4', recursive=True)
if not mp4s:
    print('❌ No mp4 found. Check Cell 6 output above.')
else:
    FINAL = mp4s[0]
    size = os.path.getsize(FINAL) // 1024
    print(f'🎥 Final video: {FINAL} ({size}KB)\n')

    with open(FINAL, 'rb') as f:
        data_url = 'data:video/mp4;base64,' + b64encode(f.read()).decode()
    display(HTML(f'<video controls width=640><source src="{data_url}" type="video/mp4"></video>'))

    print('\n⬇️ Downloading talking_monk.mp4...')
    files.download(FINAL)

## 🧾 Notes for Charlie

- **Pipeline**: Monk photo → SadTalker (audio lip-sync) → LivePortrait (motion refinement) → final video
- **Photo**: Generated via Gemini WebAPI — elderly Buddhist monk, saffron robes, temple setting
- **Audio**: Upload your own .wav/.mp3, or a placeholder gTTS audio is generated
- **Quality**: ~75-80% HeyGen for head motion + lip-sync. Body stays still (head/face only).
- **Colab timeout**: free tier disconnects after ~90 min idle. Re-running is fast (weights cached).
- **No files deleted on Mac** — everything runs in Colab's `/content/`.